# Chapter 12: Design a Chat System
**From:** *System Design Interview*

---

## TL;DR

- A chat system supports **1-on-1 chat**, **small group chat** (max 100 members), and **online presence indicators**.
- **WebSocket** is the protocol of choice for bidirectional real-time messaging, replacing polling and long polling.
- The architecture separates **stateless services** (login, signup via HTTP) from the **stateful chat service** (persistent WebSocket connections).
- **Service discovery** (e.g., Zookeeper) assigns each client to the best chat server based on geography and capacity.
- Chat history is stored in a **key-value store** (HBase/Cassandra) for horizontal scaling and low-latency access.
- **Message sync** across devices uses a per-device `cur_max_message_id` to pull only new messages.
- **Online presence** uses a heartbeat mechanism with pub-sub fanout to friends.

---
## Requirements

| Requirement | Value |
|---|---|
| Chat types | 1-on-1 + group (max 100 members) |
| Platforms | Mobile + Web |
| DAU | 50 million |
| Features | Text messages, online presence, multi-device |
| Message size limit | 100,000 characters |
| Encryption | Not required initially |
| History retention | Forever |

---
## Back-of-the-Envelope Estimation

In [ ]:
# Chat System Estimation
dau = 50_000_000  # 50M daily active users
msgs_per_user_per_day = 40  # average messages sent

total_msgs_per_day = dau * msgs_per_user_per_day
msg_qps = total_msgs_per_day / (24 * 3600)
peak_msg_qps = msg_qps * 5  # peak multiplier

# Storage: average message size ~100 bytes
avg_msg_bytes = 100
daily_storage_gb = (total_msgs_per_day * avg_msg_bytes) / (1024**3)
yearly_storage_tb = daily_storage_gb * 365 / 1024

# Connection memory: 10KB per WebSocket connection
concurrent_users = dau * 0.1  # 10% concurrent
mem_per_conn_kb = 10
total_conn_mem_gb = (concurrent_users * mem_per_conn_kb) / (1024**2)

print(f"DAU:                      {dau:>16,}")
print(f"Messages/day:             {total_msgs_per_day:>16,}")
print(f"Message QPS:              {msg_qps:>16,.0f}")
print(f"Peak QPS (5x):            {peak_msg_qps:>16,.0f}")
print(f"Daily storage:            {daily_storage_gb:>16,.1f} GB")
print(f"Yearly storage:           {yearly_storage_tb:>16,.1f} TB")
print(f"Concurrent connections:   {concurrent_users:>16,.0f}")
print(f"WebSocket memory:         {total_conn_mem_gb:>16,.1f} GB")

---
## High-Level Design

```mermaid
graph TB
    Client[Client App] -->|HTTP| LB[Load Balancer]
    LB --> API[API Servers - stateless]
    API --> Auth[Auth Service]
    API --> Profile[Profile Service]
    API --> SD[Service Discovery - Zookeeper]

    Client -->|WebSocket| CS[Chat Servers - stateful]
    CS --> MQ[Message Sync Queue]
    CS --> KV[(Key-Value Store - Chat History)]
    CS --> PS[Presence Servers]

    CS --> PN[Push Notification Servers]
```

### Stateless vs Stateful Services
| Service Type | Examples | Protocol |
|---|---|---|
| **Stateless** | Login, signup, user profile, service discovery | HTTP |
| **Stateful** | Chat service (persistent connections) | WebSocket |
| **Third-party** | Push notifications (APNs/FCM) | Provider SDK |

---
## Deep Dive: WebSocket vs Polling vs Long Polling

| Approach | How It Works | Drawbacks |
|---|---|---|
| **Polling** | Client periodically asks server for new messages | Wasteful; most responses empty |
| **Long Polling** | Client holds connection until new data or timeout | Sender/receiver server mismatch; no disconnect detection |
| **WebSocket** | Persistent bidirectional connection upgraded from HTTP | Requires connection management at scale |

```mermaid
sequenceDiagram
    participant C as Client
    participant S as Server
    Note over C,S: WebSocket Handshake
    C->>S: HTTP Upgrade Request
    S-->>C: 101 Switching Protocols
    Note over C,S: Bidirectional communication
    C->>S: Send message
    S->>C: Push message
    S->>C: Push message
    C->>S: Send message
```

WebSocket uses port 80/443, so firewalls generally allow it through.

## Deep Dive: Message Flows

### 1-on-1 Chat Flow
```mermaid
sequenceDiagram
    participant A as User A
    participant CS1 as Chat Server 1
    participant IDGen as ID Generator
    participant MQ as Message Sync Queue
    participant KV as Key-Value Store
    participant CS2 as Chat Server 2
    participant B as User B

    A->>CS1: Send message
    CS1->>IDGen: Get message_id
    CS1->>MQ: Enqueue message
    CS1->>KV: Persist message
    alt User B is online
        MQ->>CS2: Forward message
        CS2->>B: Deliver via WebSocket
    else User B is offline
        MQ->>PN: Send push notification
    end
```

### Small Group Chat Flow
- Message is copied to each group member's **message sync queue** (inbox)
- Each client only checks its own inbox for new messages
- Acceptable for small groups (up to ~500 members, per WeChat's approach)
- For large groups, this copy-per-member approach does not scale

## Deep Dive: Message Sync Across Devices

Each device maintains a `cur_max_message_id` variable:

```mermaid
graph LR
    subgraph User A Devices
        Phone[Phone - cur_max = 650]
        Laptop[Laptop - cur_max = 645]
    end
    Phone -->|fetch msg_id > 650| KV[(KV Store)]
    Laptop -->|fetch msg_id > 645| KV
```

New messages are those where:
1. `recipient_id` = current user
2. `message_id` > device's `cur_max_message_id`

## Deep Dive: Online Presence

### Heartbeat Mechanism
```mermaid
sequenceDiagram
    participant C as Client
    participant PS as Presence Server
    participant KV as KV Store

    loop Every 5 seconds
        C->>PS: Heartbeat
        PS->>KV: Update last_active_at
    end
    Note over C,PS: Client disconnects
    Note over PS: No heartbeat for 30s
    PS->>KV: Mark user offline
```

### Status Fanout via Pub-Sub
- Each friend pair has a dedicated channel (e.g., channel A-B)
- When User A goes offline, event is published to channels A-B, A-C, A-D
- Friends B, C, D subscribe to their respective channels
- For large groups (100K+), fetch status on-demand instead of pushing

## Deep Dive: Storage Choice

| Factor | Relational DB | Key-Value Store |
|---|---|---|
| **Use case** | User profiles, settings, friends | Chat messages |
| **Scale** | Moderate | Massive (60B messages/day at Facebook) |
| **Access pattern** | Varied queries | Recent-heavy with rare random access |
| **Read:Write ratio** | Read-heavy | ~1:1 for 1-on-1 chat |
| **Horizontal scaling** | Complex (sharding) | Native |
| **Examples** | MySQL, PostgreSQL | HBase (Facebook), Cassandra (Discord) |

### Message ID Generation

| Strategy | Pros | Cons |
|---|---|---|
| Auto-increment (MySQL) | Simple | Not available in most NoSQL |
| Global Snowflake ID | Unique + time-sortable | Complex distributed coordination |
| Local sequence per channel | Simple; sufficient for ordering within a conversation | IDs not globally unique |

---
## Trade-offs

| Decision | Option A | Option B | Recommendation |
|---|---|---|---|
| Protocol | HTTP long polling | WebSocket | WebSocket -- bidirectional, lower latency |
| Chat storage | Relational DB | Key-Value store | KV store -- horizontal scaling + low latency |
| Group messaging | Copy to each inbox | Single store + fan-out read | Copy for small groups; fan-out read for large |
| Presence | Poll-based | Heartbeat + pub-sub | Heartbeat -- efficient for persistent connections |
| ID generation | Global Snowflake | Local per-channel | Local -- simpler; global ordering rarely needed |
| Service discovery | Manual config | Zookeeper | Zookeeper -- dynamic assignment based on load |

---
## Key Takeaways

1. **WebSocket is the standard for real-time chat.** It provides bidirectional, persistent connections that eliminate the overhead of polling and long polling.

2. **Separate stateless and stateful services.** HTTP for login/profile, WebSocket for chat. This allows independent scaling of each tier.

3. **Key-value stores are battle-tested for chat.** HBase (Facebook Messenger) and Cassandra (Discord) handle billions of messages with low-latency access patterns.

4. **Heartbeat-based presence is robust.** Periodic heartbeats handle network flakiness gracefully, avoiding rapid online/offline flapping during brief disconnections.

5. **Message sync via cur_max_message_id is elegant.** Each device independently tracks its sync position, pulling only new messages from the KV store.

6. **Service discovery (Zookeeper) manages stateful connections.** It assigns clients to optimal chat servers and handles rebalancing when servers go down.

7. **Small group chat copies messages to per-user inboxes.** This simplifies sync but does not scale beyond a few hundred members.

---
## See Also

- [[websocket-protocol]] -- WebSocket bidirectional protocol for real-time messaging
- [[polling-long-polling]] -- Polling and long polling as WebSocket predecessors
- [[service-discovery]] -- Zookeeper-based chat server assignment
- [[message-sync]] -- Multi-device synchronization via cur_max_message_id
- [[online-presence]] -- Heartbeat mechanism and pub-sub status fanout
- [[chat-storage-kv]] -- Key-value stores for chat history persistence
- [[message-id-generation]] -- Snowflake and local sequence ID strategies